# CPS for time series: decisions and benchmarks

Examples for continuous and discrete panel targets. Both use series-specific, horizon-wise CPS calibration. The discrete demand example also uses the complete predictive distribution in the Newsvendor solver and compares the resulting decision against economic-loss and tail-risk benchmarks.

In [16]:
import os
import sys
sys.path.append(os.path.abspath("../.."))

import numpy as np
import pandas as pd
from mlforecast import MLForecast
from sklearn.ensemble import RandomForestRegressor

from tinyconformal.series import (
    ContinuousTimeSeriesConformalPredictiveSystem,
    DiscreteTimeSeriesConformalPredictiveSystem,
)
from tinyconformal.utils import NewsvendorSolver

## Demand panel

The synthetic panel avoids external dependencies and includes seasonality, trend, and overdispersion. The final 12 months form the test set.

In [17]:
rng = np.random.default_rng(42)
n_series, periods, horizon = 4, 180, 12
dates = pd.date_range("2011-01-01", periods=periods, freq="MS")
rows = []
for item in range(n_series):
    t = np.arange(periods)
    mean = 28 + 5 * item + 0.12 * t + 8 * np.sin(2 * np.pi * t / 12 + item / 3)
    demand = rng.poisson(np.maximum(mean, 1))
    rows.append(pd.DataFrame({"unique_id": f"sku_{item}", "ds": dates, "y": demand}))
df = pd.concat(rows, ignore_index=True)
train = df.groupby("unique_id", group_keys=False).head(periods - horizon).reset_index(drop=True)
test = df.groupby("unique_id", group_keys=False).tail(horizon).reset_index(drop=True)
test.head()

,unique_id,ds,y
0,sku_0,2025-01-01,54
1,sku_0,2025-02-01,47
2,sku_0,2025-03-01,59
3,sku_0,2025-04-01,47
4,sku_0,2025-05-01,59


## Discrete CPS and predictive distribution

Calibration stores signed residuals separately for each forecast-horizon step. The seasonal benchmark repeats demand observed 12 months earlier.

In [18]:
learner = MLForecast(
    models={"RandomForest": RandomForestRegressor(n_estimators=150, min_samples_leaf=3, random_state=42, n_jobs=-1)},
    freq="MS",
    lags=[1, 2, 3, 6, 12],
)
cps = DiscreteTimeSeriesConformalPredictiveSystem(
    learner=learner,
    dispersion_learner=RandomForestRegressor(
        n_estimators=150, 
        min_samples_leaf=3, 
        random_state=42, 
        n_jobs=-1,
        max_depth=int(np.ceil(np.log2(len(train)) - 1)),
        ),
    horizon=horizon, 
    n_windows=10,
    alpha=0.10, 
    minimum=0,
    nexcp=True,
).fit(train, static_features=[], n_jobs=1)
distribution = cps.predict_distribution(h=horizon)
forecast = distribution.ppf([0.10, 0.50, 0.90]).rename(columns={
    "RandomForest-q-10": "q10_cps",
    "RandomForest-q-50": "q50_cps",
    "RandomForest-q-90": "q90_cps",
})
forecast.head()

,unique_id,ds,RandomForest,q10_cps,q50_cps,q90_cps
0,sku_0,2025-01-01,47.331304,46,54,68
1,sku_0,2025-02-01,46.031679,43,49,71
2,sku_0,2025-03-01,45.361187,33,46,59
3,sku_0,2025-04-01,55.414026,42,64,71
4,sku_0,2025-05-01,52.869654,45,53,62


In [19]:
cps.evaluate(test, h=horizon)

,model,level,alpha,coverage_rate,interval_width_mean,mwis
0,RandomForest,90%,0.1,0.875,29.354,41.021


## Continuous CPS

For a continuous target, the CPS retains real-valued support. This synthetic sensor panel has series-specific levels, trend, seasonality, and Gaussian noise. As in the discrete example, calibration residuals are kept separately for every series and forecast horizon.

In [20]:
continuous_rows = []
for sensor in range(n_series):
    t = np.arange(periods)
    signal = (
        18.0
        + 1.5 * sensor
        + 0.03 * t
        + 3.5 * np.sin(2 * np.pi * t / 12 + sensor / 4)
        + rng.normal(0.0, 1.2 + 0.2 * sensor, periods)
    )
    continuous_rows.append(
        pd.DataFrame({"unique_id": f"sensor_{sensor}", "ds": dates, "y": signal})
    )
continuous_df = pd.concat(continuous_rows, ignore_index=True)
continuous_train = (
    continuous_df.groupby("unique_id", group_keys=False)
    .head(periods - horizon)
    .reset_index(drop=True)
)
continuous_test = (
    continuous_df.groupby("unique_id", group_keys=False)
    .tail(horizon)
    .reset_index(drop=True)
)
continuous_test.head()

,unique_id,ds,y
0,sensor_0,2025-01-01,22.342087
1,sensor_0,2025-02-01,22.958777
2,sensor_0,2025-03-01,26.674711
3,sensor_0,2025-04-01,26.090036
4,sensor_0,2025-05-01,24.900206


### Predictive distributions, quantiles, and intervals

`predict_distribution` exposes the row-aligned CPS object. Its `cdf`, `ppf`, and `interval` methods return results in the Nixtla long format; discrete forecasts also expose `pmf`.

In [21]:
continuous_learner = MLForecast(
    models={
        "RandomForest": RandomForestRegressor(
            n_estimators=200, min_samples_leaf=20, random_state=42, n_jobs=-1
        )
    },
    freq="MS",
    lags=[1, 2, 3, 6, 12],
)
continuous_cps = ContinuousTimeSeriesConformalPredictiveSystem(
    learner=continuous_learner,
    dispersion_learner=RandomForestRegressor(
        n_estimators=150, 
        min_samples_leaf=3, 
        random_state=42, 
        n_jobs=-1, 
        max_depth=int(np.ceil(np.log2(len(train)) - 1)),
        ),
    horizon=horizon,
    n_windows=10, 
    alpha=0.20,
    weighted_refit=False,
    nexcp=False
).fit(continuous_train, static_features=[], n_jobs=1)
continuous_distribution = continuous_cps.predict_distribution(
    h=horizon
)
continuous_points = continuous_distribution.to_frame()
continuous_forecast = continuous_distribution.cdf(
    continuous_points["RandomForest"].to_numpy()[:, None]
).rename(columns={"RandomForest-cdf": "cdf_at_point_forecast"})
continuous_forecast.head()

,unique_id,ds,RandomForest,cdf_at_point_forecast
0,sensor_0,2025-01-01,22.572876,0.454545
1,sensor_0,2025-02-01,22.181398,0.636364
2,sensor_0,2025-03-01,24.818775,0.272727
3,sensor_0,2025-04-01,24.877993,0.272727
4,sensor_0,2025-05-01,25.655589,0.454545


In [22]:
continuous_quantiles = continuous_distribution.ppf([0.10, 0.50, 0.90])
continuous_intervals = continuous_distribution.interval(coverage=0.90)
continuous_summary = continuous_quantiles.merge(
    continuous_intervals[
        ["unique_id", "ds", "RandomForest-lo-90", "RandomForest-hi-90"]
    ],
    on=["unique_id", "ds"],
)
continuous_summary.head()

,unique_id,ds,RandomForest,RandomForest-q-10,RandomForest-q-50,RandomForest-q-90,RandomForest-lo-90,RandomForest-hi-90
0,sensor_0,2025-01-01,22.572876,20.923966,23.090838,24.519941,19.583052,24.519941
1,sensor_0,2025-02-01,22.181398,21.096684,22.052022,23.806786,20.706369,23.806786
2,sensor_0,2025-03-01,24.818775,24.273502,25.321586,26.721378,24.197657,26.721378
3,sensor_0,2025-04-01,24.877993,23.150497,25.760705,27.662110,23.131499,27.662110
4,sensor_0,2025-05-01,25.655589,21.870284,26.004552,27.942595,21.557296,27.942595


In [23]:
continuous_cps.evaluate(continuous_test, h=horizon)

,model,level,alpha,coverage_rate,interval_width_mean,mwis
0,RandomForest,80%,0.2,0.771,5.275,7.996


In [24]:
distribution

## Newsvendor solver and marginal benefit

With an underage cost of 8 and an overage cost of 2, the critical fractile is 80%. `optimize_distribution` queries the conformal PPF directly, without interpolating interval endpoints. For discrete demand, `marginal_benefit_distribution` uses the conformal CDF to measure the expected net value of adding each candidate inventory unit.

In [25]:
decision = NewsvendorSolver.optimize_distribution(
    distribution, underage_cost=8.0, overage_cost=2.0
)
decision["q50_cps"] = forecast["q50_cps"]
decision["Optimal CPS"] = decision.pop("y_optimal")
decision["CPS median"] = decision["q50_cps"]
decision["Random Forest"] = np.maximum(np.rint(decision["RandomForest"]), 0)
decision["Seasonal naive"] = train.groupby("unique_id")["y"].tail(horizon).to_numpy()
decision["y"] = test["y"].to_numpy()
display(decision[["unique_id", "ds", "critical_ratio", "Optimal CPS", "CPS median", "Random Forest", "Seasonal naive", "y"]].head())
marginal_benefit = NewsvendorSolver.marginal_benefit_distribution(
    distribution,
    underage_cost=8.0,
    overage_cost=2.0,
    units=[30, 40, 50, 60],
)
marginal_benefit[["unique_id", "ds", "MB(k=30)", "MB(k=40)", "MB(k=50)", "MB(k=60)"]].head()

,unique_id,ds,critical_ratio,Optimal CPS,CPS median,Random Forest,Seasonal naive,y
0,sku_0,2025-01-01,0.8,64.0,54,47.0,52,54
1,sku_0,2025-02-01,0.8,55.0,49,46.0,48,47
2,sku_0,2025-03-01,0.8,59.0,46,45.0,45,59
3,sku_0,2025-04-01,0.8,70.0,64,55.0,69,47
4,sku_0,2025-05-01,0.8,60.0,53,53.0,52,59


,unique_id,ds,MB(k=30),MB(k=40),MB(k=50),MB(k=60)
0,sku_0,2025-01-01,8.0,7.025219,3.998990,1.999596
1,sku_0,2025-02-01,8.0,8.000000,2.964521,-0.985233
2,sku_0,2025-03-01,8.0,5.969544,0.984121,-1.005427
3,sku_0,2025-04-01,8.0,8.000000,6.995381,4.020196
4,sku_0,2025-05-01,8.0,8.000000,6.059990,1.004417


## Economic loss and tail risk

The per-period loss is $c_u\max(y-q,0)+c_o\max(q-y,0)$. Tail risk reports empirical 95% VaR and CVaR, as well as the worst-case loss.

In [26]:
models = ["Optimal CPS", "CPS median", "Random Forest", "Seasonal naive"]

def period_loss(frame, model):
    actual = frame["y"].to_numpy(float)
    order = frame[model].to_numpy(float)
    cu = frame["underage_cost"].to_numpy(float)
    co = frame["overage_cost"].to_numpy(float)
    return cu * np.maximum(actual - order, 0) + co * np.maximum(order - actual, 0)

def economic_loss(frame, model_names):
    return pd.DataFrame({"model": model_names, "economic_loss": [period_loss(frame, m).sum() for m in model_names]}).sort_values("economic_loss")

def tail_risk(frame, model_names, risk_level=0.95):
    rows = []
    for model in model_names:
        loss = period_loss(frame, model)
        var = np.quantile(loss, risk_level)
        tail = loss[loss >= var]
        rows.append({"model": model, "expected_loss": loss.mean(), "VaR_95": var, "CVaR_95": tail.mean(), "worst_loss": loss.max()})
    return pd.DataFrame(rows).sort_values("CVaR_95")

In [27]:
decision["underage_cost"] = 8.0
decision["overage_cost"] = 2.0

In [28]:
economic_loss(decision, models)

,model,economic_loss
0,Optimal CPS,1256.0
1,CPS median,1476.0
2,Random Forest,1816.0
3,Seasonal naive,2048.0


In [29]:
tail_risk(decision, models)

,model,expected_loss,VaR_95,CVaR_95,worst_loss
0,Optimal CPS,26.166667,55.3,81.333333,128.0
1,CPS median,30.750000,93.2,133.333333,200.0
2,Random Forest,37.833333,117.2,152.000000,200.0
3,Seasonal naive,42.666667,122.4,181.333333,216.0
